# 01 — Feature Engineering & Baseline LSTM

This notebook is the entry point of the pipeline. It does two things:

1. **Uses the engineered feature set** already present in `data/processed/NVDA_features.csv`
   (`date, open, high, low, close, volume` plus the indicator columns). If any indicator column
   is missing, it's computed from the raw OHLCV columns and **appended back into that same
   CSV** — this notebook never writes a separate `features.csv`.
2. **Trains a single baseline LSTM** on the *whole* dataset (no regime splitting) to predict
   next-day return, and plots **actual vs predicted price** over time.

This baseline matters for two reasons: it's the simplest possible model to sanity-check the
feature set and training loop against, and it's the fallback model used later whenever a
regime-specific model isn't available (too little history for that regime, etc.).

**Output of this notebook:** `models/baseline_lstm.pt` (trained weights) and
`reports/baseline_results.pkl` (training history + test predictions), both consumed by the
next two notebooks. All figures are saved into `reports/`.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

# ============================================================================
# Constants — everything that configures this notebook lives here.
# ============================================================================
DATA_PATH   = "data/processed/NVDA_features.csv"   # canonical engineered-feature CSV
MODELS_DIR  = "models"                             # trained model weights go here
REPORTS_DIR = "reports"                            # figures / summaries / results go here

REQUIRED_OHLCV = ["date", "open", "high", "low", "close", "volume"]
FINAL_HEADER = (
    "date,open,high,low,close,volume,return_1,return_5,log_return_1,sma_20,sma_50,"
    "sma_200,sma_ratio_20,sma_ratio_50,ema_12,ema_20,ema_26,ema_ratio_20,rsi_14,macd,"
    "macd_signal,macd_hist,bb_upper,bb_lower,bb_width,bb_pct_b,roc_5,roc_10,atr_14,"
    "atr_ratio,stoch_k,stoch_d,obv,obv_change,vwap,vwap_ratio,z_score_20,future_return,"
    "target_direction"
).split(",")

FEATURE_COLUMNS = [
    "return_1", "return_5", "log_return_1",
    "sma_ratio_20", "sma_ratio_50", "ema_ratio_20",
    "rsi_14", "macd", "macd_signal", "macd_hist",
    "bb_width", "bb_pct_b",
    "roc_5", "roc_10",
    "atr_ratio",
    "stoch_k", "stoch_d",
    "obv_change",
    "vwap_ratio",
    "z_score_20",
]

SEQUENCE_LENGTH = 5      # 20 trading days of history per prediction
TRAIN_SPLIT     = 0.80    # 80% train+val, 20% held-out test
VAL_FRACTION    = 0.15    # 15% of the train portion becomes validation
SEED            = 42

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

# Dark background theme for every figure produced in this notebook.
plt.style.use("dark_background")
plt.rcParams.update({
    "figure.facecolor":  "#0d1117",
    "axes.facecolor":    "#0d1117",
    "savefig.facecolor": "#0d1117",
    "axes.grid": True,
    "grid.alpha": 0.25,
})

torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cpu


## 1. Load raw price data

`DATA_PATH` (set above in the constants cell) points at `data/processed/NVDA_features.csv`.
Only `date, open, high, low, close, volume` are required as a minimum — if every indicator
column listed in `FINAL_HEADER` is already present in that file, it's used as-is and nothing
gets recomputed; otherwise the missing indicators are computed from those six raw columns and
written back into `DATA_PATH` (see Section 2).


In [ ]:
raw = pd.read_csv("/content/AAPL (2).csv")
raw["date"] = pd.to_datetime(raw["date"])
raw = raw.sort_values("date").reset_index(drop=True)

missing = [c for c in REQUIRED_OHLCV if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required OHLCV columns: {missing}")

print(f"Loaded {len(raw)} rows from {DATA_PATH} | {raw['date'].min().date()} -> {raw['date'].max().date()}")
raw[REQUIRED_OHLCV].head()


Loaded 2858 rows from data/processed/NVDA_features.csv | 2015-01-02 -> 2026-05-14


,date,open,high,low,close,volume
0,2015-01-02,27.847500,27.860001,26.837500,27.332500,212818400
1,2015-01-05,27.072500,27.162500,26.352501,26.562500,257142000
2,2015-01-06,26.635000,26.857500,26.157499,26.565001,263188400
3,2015-01-07,26.799999,27.049999,26.674999,26.937500,160423600
4,2015-01-08,27.307501,28.037500,27.174999,27.972500,237458000


## 2. Indicator calculation

> **Before writing any code — run the cell below as-is first.**
> It will automatically detect which columns (if any) are missing from your dataset.
>
> - **If your dataset is already complete** (all 39 columns present), the cell will print `"Dataset complete — skipping indicator calculation."` and exit early. You do not need to implement anything.
> - **If columns are missing**, the cell will print exactly which ones. Implement only the `# TODO` blocks that correspond to those missing indicators — leave the rest untouched.

The six indicator groups you may need to implement (only if flagged as missing):

1. **RSI (14-period)** — 14-period Relative Strength Index scaled to **0–1**.  
   Use rolling average gains and losses from `close.diff()`, then apply the classic RS formula.

2. **MACD family** — `macd` (EMA12 − EMA26), `macd_signal` (9-period EMA of MACD), `macd_hist` (their difference).  
   `ema_12` and `ema_26` are already computed above each TODO block.

3. **Bollinger Bands** — `bb_upper`, `bb_lower`, `bb_width` (band width / SMA), `bb_pct_b` (price position 0–1).  
   Use `sma_20` (already computed) and a 20-day rolling std.

4. **Stochastic Oscillator (14, 3)** — `stoch_k` from the 14-period high/low range, `stoch_d` as its 3-period rolling mean. Scale **both to 0–1**.

5. **On-Balance Volume** — `obv` (cumulative signed volume), `obv_change` (percentage change of OBV).

6. **VWAP & ratio** — Cumulative `vwap`, and `vwap_ratio = close / vwap − 1`.


In [ ]:
# ── Step 1: detect which required columns are already present ────────────────
missing_cols = [col for col in FINAL_HEADER if col not in raw.columns]

if not missing_cols:
    print("Dataset complete — skipping indicator calculation.")
    print(f"All {len(FINAL_HEADER)} required columns found. Loading directly.")
    featured = raw[FINAL_HEADER].copy()
else:
    print(f"{len(missing_cols)} column(s) missing from your dataset: {missing_cols}")
    print("Computing missing indicators...\n")

    def calculate_indicators(df: pd.DataFrame) -> pd.DataFrame:
        """Compute only the missing indicator columns and merge them back."""

        df = df.copy().sort_values("date").reset_index(drop=True)

        close = df["close"]
        high = df["high"]
        low = df["low"]
        volume = df["volume"]

        # --- Returns ---
        df["return_1"] = close.pct_change(1)
        df["return_5"] = close.pct_change(5)
        df["log_return_1"] = np.log(close / close.shift(1))

        # --- Moving averages & trend ratios ---
        df["sma_20"] = close.rolling(20).mean()
        df["sma_50"] = close.rolling(50).mean()
        df["sma_200"] = close.rolling(200).mean()

        df["sma_ratio_20"] = close / df["sma_20"] - 1
        df["sma_ratio_50"] = close / df["sma_50"] - 1

        df["ema_12"] = close.ewm(span=12, adjust=False).mean()
        df["ema_20"] = close.ewm(span=20, adjust=False).mean()
        df["ema_26"] = close.ewm(span=26, adjust=False).mean()

        df["ema_ratio_20"] = close / df["ema_20"] - 1

        # --- RSI(14) ---
        delta = close.diff()

        gain = delta.where(delta > 0, 0)
        loss = -delta.where(delta < 0, 0)

        avg_gain = gain.rolling(14).mean()
        avg_loss = loss.rolling(14).mean()

        rs = avg_gain / avg_loss
        df["rsi_14"] = (100 - (100 / (1 + rs))) / 100

        # --- MACD ---
        df["macd"] = df["ema_12"] - df["ema_26"]
        df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
        df["macd_hist"] = df["macd"] - df["macd_signal"]

        # --- Bollinger Bands ---
        rolling_std = close.rolling(20).std()

        df["bb_upper"] = df["sma_20"] + 2 * rolling_std
        df["bb_lower"] = df["sma_20"] - 2 * rolling_std

        df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["sma_20"]
        df["bb_pct_b"] = (close - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

        # --- Rate of Change ---
        df["roc_5"] = close.pct_change(5)
        df["roc_10"] = close.pct_change(10)

        # --- ATR(14) ---
        prev_close = close.shift(1)

        tr = pd.concat([
            high - low,
            (high - prev_close).abs(),
            (low - prev_close).abs()
        ], axis=1).max(axis=1)

        df["atr_14"] = tr.rolling(14).mean()
        df["atr_ratio"] = df["atr_14"] / close

        # --- Stochastic Oscillator ---
        lowest_low = low.rolling(14).min()
        highest_high = high.rolling(14).max()

        df["stoch_k"] = (close - lowest_low) / (highest_high - lowest_low)
        df["stoch_d"] = df["stoch_k"].rolling(3).mean()

        # --- On Balance Volume ---
        direction = np.sign(close.diff()).fillna(0)

        df["obv"] = (direction * volume).cumsum()
        df["obv_change"] = df["obv"].pct_change()

        # --- VWAP ---
        typical_price = (high + low + close) / 3

        df["vwap"] = (typical_price * volume).cumsum() / volume.cumsum()
        df["vwap_ratio"] = close / df["vwap"] - 1

        # --- Z-score ---
        roll_std = close.rolling(20).std()
        df["z_score_20"] = (close - df["sma_20"]) / roll_std

        # --- Targets ---
        df["future_return"] = close.shift(-1) / close - 1
        df["target_direction"] = (df["future_return"] > 0).astype(int)

        return df

    featured = calculate_indicators(raw[REQUIRED_OHLCV])
    featured = featured[FINAL_HEADER]

    # Save processed dataset safely
    import os

    save_path = DATA_PATH

    # If the directory doesn't exist, create it.
    directory = os.path.dirname(save_path)

    if directory:
        os.makedirs(directory, exist_ok=True)

    featured.to_csv(save_path, index=False)

    print(f"Missing columns computed and saved to {save_path}")

featured = featured.dropna().reset_index(drop=True)

print(f"Final feature set: {len(featured)} rows, {len(featured.columns)} columns")

featured.head()

33 column(s) missing from your dataset: ['return_1', 'return_5', 'log_return_1', 'sma_20', 'sma_50', 'sma_200', 'sma_ratio_20', 'sma_ratio_50', 'ema_12', 'ema_20', 'ema_26', 'ema_ratio_20', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_lower', 'bb_width', 'bb_pct_b', 'roc_5', 'roc_10', 'atr_14', 'atr_ratio', 'stoch_k', 'stoch_d', 'obv', 'obv_change', 'vwap', 'vwap_ratio', 'z_score_20', 'future_return', 'target_direction']
Computing missing indicators...

Missing columns computed and saved to data/processed/NVDA_features.csv
Final feature set: 2658 rows, 39 columns


,date,open,high,low,close,volume,return_1,return_5,log_return_1,sma_20,...,atr_ratio,stoch_k,stoch_d,obv,obv_change,vwap,vwap_ratio,z_score_20,future_return,target_direction
0,2015-10-16,27.945000,28.0000,27.632500,27.760000,156930400,-0.007331,-0.009633,-0.007358,27.942375,...,0.022881,0.601613,0.512501,-966810800.0,0.193770,30.080762,-0.077151,-0.387890,0.006214,1
1,2015-10-19,27.700001,27.9375,27.527500,27.932501,119036800,0.006214,0.001165,0.006195,27.898875,...,0.020176,0.812501,0.680279,-847774000.0,-0.123123,30.074521,-0.071224,0.079227,0.018258,1
2,2015-10-20,27.834999,28.5425,27.705000,28.442499,195871200,0.018258,0.017712,0.018094,27.903500,...,0.020153,0.941691,0.785268,-651902800.0,-0.231042,30.066256,-0.054006,1.253292,-0.000088,0
3,2015-10-21,28.500000,28.8950,28.424999,28.440001,167180800,-0.000088,0.032211,-0.000088,27.896500,...,0.019458,0.773350,0.842514,-819083600.0,0.256451,30.060619,-0.053912,1.295636,0.015295,1
4,2015-10-22,28.582500,28.8750,28.525000,28.875000,166616400,0.015295,0.032541,0.015180,27.902750,...,0.018101,0.989145,0.901395,-652467200.0,-0.203418,30.055693,-0.039283,2.242449,0.030996,1


## 3. Feature columns, scaling, and sequence construction

The model never sees raw prices directly — only the ratio/percentage-style derived features in `FEATURE_COLUMNS`.

**Your tasks:**

1. **`make_sequences(features, targets, seq_len)`** — Given a 2-D array of features and a 1-D array of targets, build overlapping windows.  
   For each index `i` from `seq_len` to `len(features)`, the input window is `features[i - seq_len : i]` and the label is `targets[i]`.  
   Return two NumPy arrays of dtype `float32`: `X` with shape `(N, seq_len, n_features)` and `y` with shape `(N,)`.

2. **`winsorise(arr, n_std=3.0)`** — Clip extreme values in a 1-D return array.  
   Compute a symmetric range of `±(n_std × arr.std())` and clip values outside it. Return the clipped array.

The scaler fitting, splitting, and sequence-building calls below are already written — you only need to implement the two functions above.


In [ ]:
print(f"{len(FEATURE_COLUMNS)} input features:", FEATURE_COLUMNS)


def make_sequences(features: np.ndarray, targets: np.ndarray, seq_len: int):
    """
    Build (X, y) pairs from a scaled feature array and a target array.

    For each position i in [seq_len, len(features)), produce:
      X[i] = features[i - seq_len : i]   shape: (seq_len, n_features)
      y[i] = targets[i]

    Returns two float32 NumPy arrays: X of shape (N, seq_len, n_features)
    and y of shape (N,).
    """

    xs = []
    ys = []

    for i in range(seq_len, len(features)):
        xs.append(features[i - seq_len:i])
        ys.append(targets[i])

    X = np.array(xs, dtype=np.float32)
    y = np.array(ys, dtype=np.float32)

    return X, y


def winsorise(arr: np.ndarray, n_std: float = 3.0) -> np.ndarray:
    """
    Clip values in arr that fall outside ±(n_std * arr.std()).
    Applied only to training targets so test evaluation stays unmodified.
    """

    mean = np.mean(arr)
    std = np.std(arr)

    lo = mean - n_std * std
    hi = mean + n_std * std

    return np.clip(arr, lo, hi)


split_idx = int(len(featured) * TRAIN_SPLIT)
val_size = max(int(split_idx * VAL_FRACTION), 1)
train_end = split_idx - val_size

scaler = StandardScaler()
scaler.fit(featured.iloc[:train_end][FEATURE_COLUMNS])
all_feats = scaler.transform(featured[FEATURE_COLUMNS])

raw_targets = featured["future_return"].to_numpy(dtype=np.float32)
train_targets = winsorise(raw_targets[:train_end])
targets = np.concatenate([train_targets, raw_targets[train_end:]])

x_all, y_all = make_sequences(all_feats, targets, SEQUENCE_LENGTH)

n_train = max(train_end - SEQUENCE_LENGTH, 1)
n_val = max(split_idx - SEQUENCE_LENGTH - n_train, 0)

x_train, y_train = x_all[:n_train], y_all[:n_train]
x_val, y_val = x_all[n_train:n_train + n_val], y_all[n_train:n_train + n_val]
x_test, y_test = x_all[n_train + n_val:], y_all[n_train + n_val:]

print(
    f"train={len(x_train)}  val={len(x_val)}  test={len(x_test)}  "
    f"(sequence length = {SEQUENCE_LENGTH})"
)


20 input features: ['return_1', 'return_5', 'log_return_1', 'sma_ratio_20', 'sma_ratio_50', 'ema_ratio_20', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_width', 'bb_pct_b', 'roc_5', 'roc_10', 'atr_ratio', 'stoch_k', 'stoch_d', 'obv_change', 'vwap_ratio', 'z_score_20']
train=1803  val=318  test=532  (sequence length = 5)


## 4. Model: LSTM with temporal attention

A plain LSTM only uses its **last** hidden state, which can miss signals from earlier in the window. **Temporal attention** learns a weighting over all time steps, then combines them into a single context vector.

**Your tasks:**

1. **`TemporalAttention.forward(lstm_out)`** — `lstm_out` has shape `(batch, seq_len, hidden)`.  
   - Apply `self.scorer` (a single Linear layer) to get an unnormalised score per timestep: shape `(batch, seq_len, 1)`.  
   - Normalise across the **time axis** (dim=1) using softmax to get attention weights.  
   - Return the weighted sum of `lstm_out` along the time axis: shape `(batch, hidden)`.

2. **`ReturnSequenceModel.forward(x)`** — `x` has shape `(batch, seq_len, input_size)`.  
   - Pass `x` through `self.recurrent` to get `out` (all hidden states) and `_` (final state, unused).  
   - Pass `out` through `self.attention` to get a single context vector.  
   - Pass the context through `self.head` and squeeze the last dimension to return a 1-D tensor of shape `(batch,)`.


In [ ]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_size: int) -> None:
        super().__init__()
        self.scorer = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, lstm_out: torch.Tensor) -> torch.Tensor:
        """
        Args:
            lstm_out: tensor of shape (batch, seq_len, hidden_size)
        Returns:
            context: tensor of shape (batch, hidden_size)
        """
        # TODO: compute per-timestep scores using self.scorer (shape: batch, seq_len, 1)
        # TODO: normalise scores across the time axis (dim=1) via softmax
        # TODO: compute and return the weighted sum of lstm_out along dim=1
        pass


class ReturnSequenceModel(nn.Module):
    """LSTM / GRU / BiLSTM + temporal attention + small regression head."""
    def __init__(self, input_size, hidden_size=64, num_layers=2,
                 dropout=0.4, model_type="lstm") -> None:
        super().__init__()
        model_type = model_type.lower()
        bidirectional = model_type == "bilstm"
        recurrent_cls = nn.GRU if model_type == "gru" else nn.LSTM
        self.recurrent = recurrent_cls(
            input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True, bidirectional=bidirectional,
        )
        out_dim = hidden_size * (2 if bidirectional else 1)
        self.attention = TemporalAttention(out_dim)
        self.head = nn.Sequential(
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(64, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: tensor of shape (batch, seq_len, input_size)
        Returns:
            predictions: tensor of shape (batch,)
        """
        # TODO: pass x through self.recurrent — unpack the tuple to get `out` (all hidden states)
        # TODO: pass out through self.attention to get a context vector
        # TODO: pass context through self.head and squeeze the last dim
        pass


## 5. Loss function: direction-aware, class-balanced

Plain regression loss (MSE/Huber) has no notion of *sign*. This custom loss adds an explicit direction signal.

**Your task — implement `DirectionWeightedLoss.forward(pred, target)`:**

The loss has two components that are summed:

1. **Weighted Huber term** — Use `self.huber(pred, target)` (returns per-sample losses, shape `(batch,)`).  
   Build a per-sample weight tensor `w`: `self.up_weight` where `target > 0`, else `self.down_weight`.  
   Return the mean of `huber_losses * w`.

2. **Weighted BCE term** — Convert the regression predictions to a direction signal by multiplying by `self.temp` (a temperature that sharpens the soft-sigmoid).  
   Compute `binary_cross_entropy_with_logits` between `pred * self.temp` and `(target > 0).float()`, with `reduction="none"`.  
   Take the mean of `bce_losses * w` (same weights as above) and scale by `self.lambda_dir`.

Return the sum of both terms.

`compute_class_weights` is already implemented — it returns `(up_weight, down_weight)` as balanced inverse-frequency weights.


In [ ]:
class DirectionWeightedLoss(nn.Module):
    def __init__(self, up_weight=1.0, down_weight=1.0, lambda_dir=0.30, temp=50.0):
        super().__init__()
        self.up_weight = up_weight
        self.down_weight = down_weight
        self.lambda_dir = lambda_dir
        self.temp = temp
        self.huber = nn.SmoothL1Loss(reduction="none")

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """
        Args:
            pred:   predicted returns, shape (batch,)
            target: actual returns,    shape (batch,)
        Returns:
            scalar loss combining weighted Huber regression and weighted BCE direction terms
        """

        # Per-sample weights
        w = torch.where(
            target > 0,
            torch.full_like(target, self.up_weight),
            torch.full_like(target, self.down_weight)
        )

        # Weighted Huber loss
        huber_loss = (self.huber(pred, target) * w).mean()

        # Direction labels
        dir_target = (target > 0).float()

        # Binary cross-entropy for direction prediction
        bce_per_sample = F.binary_cross_entropy_with_logits(
            pred * self.temp,
            dir_target,
            reduction="none"
        )

        # Weighted BCE loss
        direction_loss = (bce_per_sample * w).mean()

        # Combined loss
        loss = huber_loss + self.lambda_dir * direction_loss

        return loss


def compute_class_weights(y_train: np.ndarray):
    """Balanced weights: N / (2 * n_class) for each direction."""
    n = len(y_train)
    n_up = max((y_train > 0).sum(), 1)
    n_down = max((y_train <= 0).sum(), 1)

    return float(n / (2.0 * n_up)), float(n / (2.0 * n_down))


## 6. Training loop

Key design choices in the training loop below:
- **`ReduceLROnPlateau`** drops the learning rate only when validation loss genuinely stalls.
- **Early stopping** on validation loss restores the best-loss checkpoint at the end.

**Your task — inside `train_model()`, complete the per-batch training step:**

Each iteration over `loader` yields `(bx, by)` (a batch of input sequences and targets). You must:
1. Move `bx` and `by` to `DEVICE`.
2. Zero the optimizer gradients.
3. Compute the model's output and the batch loss using `loss_fn`.
4. Run backpropagation.
5. Clip gradients to `config.grad_clip` using `nn.utils.clip_grad_norm_`.
6. Take an optimizer step.

The optimizer construction (`AdamW`), scheduler, validation loop, early-stopping logic, and metric collection are all already written for you.


In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 5
    batch_size: int = 64
    learning_rate: float = 0.05
    weight_decay: float = 1e-3
    grad_clip: float = 1.0
    patience: int = 10
    lambda_dir: float = 0.30
    hidden_size: int = 8
    num_layers: int = 2
    dropout: float = 0.4
    model_type: str = "lstm"


def train_model(x_train, y_train, x_val, y_val, x_test, y_test,
                config: TrainConfig, verbose=True):

    w_up, w_down = compute_class_weights(y_train)

    loss_fn = DirectionWeightedLoss(
        up_weight=w_up,
        down_weight=w_down,
        lambda_dir=config.lambda_dir,
    )

    model = ReturnSequenceModel(
        input_size=x_train.shape[-1],
        hidden_size=config.hidden_size,
        num_layers=config.num_layers,
        dropout=config.dropout,
        model_type=config.model_type,
    ).to(DEVICE)

    loader = DataLoader(
        TensorDataset(torch.tensor(x_train), torch.tensor(y_train)),
        batch_size=config.batch_size,
        shuffle=True,
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
        min_lr=config.learning_rate * 0.02,
    )

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_dir_acc": [],
        "lr": [],
    }

    best_val = float("inf")
    best_state = None
    patience_ctr = 0

    best_acc = -1.0
    best_acc_epoch = -1

    for epoch in range(config.epochs):

        model.train()
        ep_losses = []

        for bx, by in loader:

            # Move data to device
            bx = bx.to(DEVICE)
            by = by.to(DEVICE)

            # Zero gradients
            optimizer.zero_grad()

            # Forward pass
            preds = model(bx)
            loss = loss_fn(preds, by)

            # Backward pass
            loss.backward()

            # Gradient clipping
            nn.utils.clip_grad_norm_(
                model.parameters(),
                config.grad_clip,
            )

            # Update weights
            optimizer.step()

            # Store loss
            ep_losses.append(loss.item())

        train_loss = float(np.mean(ep_losses))

        model.eval()

        with torch.no_grad():
            vp = model(torch.tensor(x_val).to(DEVICE))
            vy = torch.tensor(y_val).to(DEVICE)

            val_loss = float(loss_fn(vp, vy).item())
            val_preds = vp.cpu().numpy()
            val_acc = float(
                accuracy_score(
                    y_val > 0,
                    val_preds > 0,
                )
            )

        scheduler.step(val_loss)

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = {
                k: v.clone()
                for k, v in model.state_dict().items()
            }
            patience_ctr = 0
        else:
            patience_ctr += 1

        if val_acc > best_acc:
            best_acc = val_acc
            best_acc_epoch = epoch + 1

        if patience_ctr >= config.patience:
            if verbose:
                print(
                    f"  Early stop @ epoch {epoch+1} "
                    f"(best val={best_val:.6f})"
                )
            break

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dir_acc"].append(val_acc)
        history["lr"].append(
            optimizer.param_groups[0]["lr"]
        )

        if verbose and ((epoch + 1) % 10 == 0 or epoch == 0):
            print(
                f"  Epoch {epoch+1:03d}  "
                f"train={train_loss:.6f}  "
                f"val={val_loss:.6f}  "
                f"dir_acc={val_acc:.3f}  "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

    if best_state:
        model.load_state_dict(best_state)

    if (
        verbose
        and best_acc_epoch > 0
        and best_acc_epoch < len(history["val_loss"]) - 3
    ):
        print(
            f"  [note] best val_dir_acc={best_acc:.3f} "
            f"was at epoch {best_acc_epoch}, "
            f"but the deployed checkpoint is the "
            f"lowest-val-loss one instead."
        )

    model.eval()

    with torch.no_grad():
        test_preds = model(
            torch.tensor(x_test).to(DEVICE)
        ).cpu().numpy()

    metrics = {
        "test_rmse": float(
            np.sqrt(
                mean_squared_error(
                    y_test,
                    test_preds,
                )
            )
        ),
        "test_mae": float(
            mean_absolute_error(
                y_test,
                test_preds,
            )
        ),
        "direction_accuracy": float(
            accuracy_score(
                y_test > 0,
                test_preds > 0,
            )
        ),
        "f1_score": float(
            f1_score(
                y_test > 0,
                test_preds > 0,
                zero_division=0,
            )
        ),
        "n_train": len(x_train),
        "n_val": len(x_val),
        "n_test": len(x_test),
        "y_test": y_test,
        "preds": test_preds,
    }

    return model, history, metrics


## 7. Actual vs predicted price

The model predicts next-day **return**, not price directly. To plot it against the actual price
series, we reconstruct an implied price path: `predicted_price[t+1] = actual_price[t] *
(1 + predicted_return[t+1])`. This is a one-step-ahead reconstruction — each predicted point uses
the **true** price from the day before, not a chained prediction, so errors don't compound across
the chart. It's the right way to visually sanity-check a return-prediction model against price.


In [16]:


# Starting index of the test set
test_start_idx = len(featured) - len(y_test)

# Dates corresponding to the test set
test_dates = featured["date"].iloc[test_start_idx:].reset_index(drop=True)

# Previous day's closing price (base price for return calculation)
actual_price_at_t = (
    featured["close"]
    .iloc[test_start_idx - 1 : -1]
    .reset_index(drop=True)
)

# Sanity check
assert len(test_dates) == len(actual_price_at_t) == len(baseline_metrics["y_test"]) == len(baseline_metrics["preds"])

# Reconstruct actual and predicted prices
actual_next_price = actual_price_at_t.values * (1 + baseline_metrics["y_test"])
predicted_next_price = actual_price_at_t.values * (1 + baseline_metrics["preds"])

# -----------------------------
# Plot Actual vs Predicted Prices
# -----------------------------

plt.figure(figsize=(14, 5))

plt.plot(
    test_dates,
    actual_next_price,
    label="Actual Price",
    color="#2ecc71",
    linewidth=1.5,
)

plt.plot(
    test_dates,
    predicted_next_price,
    label="Predicted Price",
    color="#e74c3c",
    linewidth=1.2,
    alpha=0.85,
)

plt.title("Baseline LSTM — Actual vs Predicted Price (Test Set)")
plt.xlabel("Date")
plt.ylabel("Stock Price ($)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

# Save the figure
plt.savefig(
    f"{REPORTS_DIR}/baseline_actual_vs_predicted_price.png",
    dpi=140,
    bbox_inches="tight",
)

plt.show()

# -----------------------------
# Optional: Display first few predictions
# -----------------------------

results_df = pd.DataFrame({
    "Date": test_dates,
    "Previous Close": actual_price_at_t,
    "Actual Price": actual_next_price,
    "Predicted Price": predicted_next_price,
    "Actual Return": baseline_metrics["y_test"],
    "Predicted Return": baseline_metrics["preds"],
})

print(results_df.head())

# Save results for later analysis
results_df.to_csv(
    f"{REPORTS_DIR}/baseline_actual_vs_predicted_price.csv",
    index=False,
)

NameError: name 'baseline_metrics' is not defined

In [17]:
# ----------------------------------------------------------------------------
# Save everything this notebook produced — model weights into MODELS_DIR, and
# the training history / metrics / test predictions (needed by the next two
# notebooks) into REPORTS_DIR. This is the only place either directory is
# written to.
# ----------------------------------------------------------------------------
torch.save(baseline_model.state_dict(), f"{MODELS_DIR}/baseline_lstm.pt")

with open(f"{MODELS_DIR}/baseline_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(f"{REPORTS_DIR}/baseline_results.pkl", "wb") as f:
    pickle.dump({
        "history": baseline_history,
        "metrics": baseline_metrics,
        "feature_columns": FEATURE_COLUMNS,
        "sequence_length": SEQUENCE_LENGTH,
    }, f)

print(f"Saved model weights  -> {MODELS_DIR}/baseline_lstm.pt")
print(f"Saved scaler         -> {MODELS_DIR}/baseline_scaler.pkl")
print(f"Saved results pickle -> {REPORTS_DIR}/baseline_results.pkl")
print("Next: open Regime_detection_02.ipynb ")


NameError: name 'baseline_model' is not defined

## Summary

- Used the engineered feature set already present in `data/processed/NVDA_features.csv` — no
  separate `features.csv` was generated; any indicator this notebook had to compute itself gets
  appended back into that same CSV instead.
- Trained one LSTM on the whole dataset using a direction-aware, class-balanced loss.
- The direction accuracy and F1 printed above are the baseline that regime-specific models in
  the next notebook need to beat to justify their extra complexity.
- Saved the trained model to `models/baseline_lstm.pt` and this run's history/metrics to
  `reports/baseline_results.pkl` for reuse downstream. The actual-vs-predicted price chart was
  saved into `reports/` using the dark theme set up at the top of the notebook.
